# MLOps & Production ML System Design

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T31SHA/data-ml-handbook/blob/main/notebooks/18_mlops.ipynb)

Runnable examples from **Data/ML Handbook** — section `18_mlops`. Read the explanations in the [docs](https://t31sha.github.io/data-ml-handbook/handbook/18_mlops/).

In [ ]:
# Setup: works in Colab or locally from the repo root
import os, subprocess, sys
if not os.path.exists("code") and not os.path.exists("../code"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/T31SHA/data-ml-handbook.git"], check=True)
    os.chdir("data-ml-handbook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
if os.path.exists("../code"): os.chdir("..")
os.chdir("code"); os.makedirs("data", exist_ok=True)
for prep in ["proj01_make_csv.py", "s02_load.py"]:          # shared datasets used by many snippets
    if not os.path.exists("data/housing.parquet"): subprocess.run([sys.executable, prep], check=True)
sys.path.insert(0, "../figures/src")

### `code/s15_mlflow.py`

In [ ]:
import mlflow
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
X, y = load_breast_cancer(return_X_y=True)
mlflow.set_experiment("cancer-baselines")
for C in [0.01, 0.1, 1.0]:
    with mlflow.start_run(run_name=f"logreg_C={C}"):
        mlflow.log_params({"model": "logreg", "C": C})
        auc = cross_val_score(LogisticRegression(C=C, max_iter=5000), X, y, cv=5, scoring="roc_auc").mean()
        mlflow.log_metric("cv_auc", auc); mlflow.set_tag("data_version", "v2026-10-24")
runs = mlflow.search_runs(order_by=["metrics.cv_auc DESC"])
print(runs[["tags.mlflow.runName", "params.C", "metrics.cv_auc"]].round(4).to_string(index=False))

### `code/proj05_serve.py`

In [ ]:
import joblib, pandas as pd
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

model = joblib.load("data/house_model.joblib")
app = FastAPI(title="house-price")
class Block(BaseModel):
    MedInc: float = Field(gt=0); HouseAge: float; AveRooms: float; AveBedrms: float
    Population: float; AveOccup: float; Latitude: float; Longitude: float; region: str
@app.post("/predict")
def predict(b: Block):
    yhat = float(model.predict(pd.DataFrame([b.model_dump()]))[0])
    return {"median_value_usd": round(yhat * 100_000), "model_version": "lgbm-v1"}

client = TestClient(app)                                  # in-process test, no server needed
r = client.post("/predict", json=dict(MedInc=8.3, HouseAge=41, AveRooms=6.98, AveBedrms=1.02, Population=322,
                                     AveOccup=2.55, Latitude=37.88, Longitude=-122.23, region="NorCal"))
print(r.status_code, r.json())
bad = client.post("/predict", json=dict(MedInc=-1, HouseAge=41, AveRooms=6.98, AveBedrms=1.02, Population=322,
                                       AveOccup=2.55, Latitude=37.88, Longitude=-122.23, region="NorCal"))
print(bad.status_code, "validation error ->", bad.json()["detail"][0]["msg"])

### `code/s13_psi.py`

In [ ]:
import numpy as np
def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected); a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(np.sum((a - e) * np.log(a / e)))
rng = np.random.default_rng(0); train = rng.normal(50, 10, 50_000)
for name, prod in [("same distribution", rng.normal(50, 10, 20_000)), ("mean +2", rng.normal(52, 10, 20_000)),
                   ("mean +5, wider", rng.normal(55, 13, 20_000))]:
    v = psi(train, prod); print(f"{name:18s} PSI={v:.3f} -> {'stable' if v < .1 else 'moderate' if v < .25 else 'SIGNIFICANT'}")
from scipy.stats import ks_2samp
print(f"KS test p (mean +2): {ks_2samp(train, rng.normal(52, 10, 20_000)).pvalue:.1e}  <- tiny p at large n: use an effect size (PSI) too")